# Lonboard Map experiments

In [1]:
import geopandas as gpd
import pandas as pd
from lonboard import Map, PathLayer, viz
from shapely import LineString

from datetime import timedelta

import movingpandas as mpd
from lonboard import TripsLayer


/home/knuthp/src/spartid-ais/.venv/lib/python3.13/site-packages/movingpandas/__init__.py:41: UserWarning: Missing optional dependencies. To use the trajectory smoother classes please install Stone Soup (see https://stonesoup.readthedocs.io/en/latest/#installation).
  warnings.warn(e.msg, UserWarning)


In [2]:
# Bounding box: Oslofjord north of Sandefjord, east to Halden, north to Oslo
min_lon, min_lat = 10.2167, 59.1313   # Sandefjord
max_lon, max_lat = 11.3875, 59.9127   # Halden (lon), Oslo (lat)

## Read in parquet file for a week

In [3]:
%%time
df_imo = pd.read_parquet("imo_vessel_codes.parquet")
len(df_imo)

CPU times: user 7.35 ms, sys: 15.9 ms, total: 23.3 ms
Wall time: 43.3 ms


20795

In [4]:
%%time
df_raw = pd.read_parquet("historic_position_2024_w15.parquet", dtype_backend="pyarrow")
df = (
    df_raw
    # .query("timestamp.dt.weekday == 1")
    .query("(@min_lon <= long <= @max_lon) and (@min_lat <= lat <= @max_lat)")
    .merge(df_imo, left_on="mmsi", right_on="mmsi")
    .assign(timestamp_str=lambda df_: df_["timestamp"].astype(str))
    #.drop(columns=["timestamp", "timestamp_1"])
    .drop(columns=["timestamp_1"])
)
len(df)
df

CPU times: user 1.82 s, sys: 2.45 s, total: 4.27 s
Wall time: 1.5 s


,year,month,week,timestamp,id,msg_type,repeat,mmsi,status,turn,...,course,heading,maneuver,raim,radio,imo,name,flag,type,timestamp_str
0,2024,4,15,2024-04-08 00:00:00.664767,43265181,1,0,257935000,UnderWayUsingEngine,0.0,...,349.4,349,NotAvailable,False,28396,9956862,SELVAAGSUND,,70,2024-04-08T00:00:00.664767
1,2024,4,15,2024-04-08 00:00:00.677092,43265191,1,0,257556800,UnderWayUsingEngine,-128.0,...,196.1,511,NotAvailable,False,2240,0,RESCUE EIVIND ECKBO,,51,2024-04-08T00:00:00.677092
2,2024,4,15,2024-04-08 00:00:03.832583,43265246,1,0,257058500,UnderWayUsingEngine,0.0,...,140.5,0,NotAvailable,False,65661,0,INGEBORG PLATOU,,52,2024-04-08T00:00:03.832583
3,2024,4,15,2024-04-08 00:00:03.843323,43265255,1,0,257319700,UnderWayUsingEngine,-128.0,...,360.0,511,NotAvailable,True,81967,0,PRINSESSE KRISTINA,,69,2024-04-08T00:00:03.843323
4,2024,4,15,2024-04-08 00:00:04.801238,43265259,1,0,257173700,UnderWayUsingEngine,0.0,...,360.0,15,NotAvailable,False,49180,0,TUROY,,69,2024-04-08T00:00:04.801238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441641,2024,4,15,2024-04-14 14:03:11.250461,53619933,1,0,259402000,UnderWayUsingEngine,0.0,...,92.0,14,NotAvailable,False,33159,9144093,BASTO II,,60,2024-04-14T14:03:11.250461
441642,2024,4,15,2024-04-14 14:03:11.257431,53619939,3,0,258010350,UnderWayUsingEngine,5.0,...,59.6,16,NotAvailable,False,3905,9907990,SVELVIK,,69,2024-04-14T14:03:11.257431
441643,2024,4,15,2024-04-14 14:03:12.273384,53619959,1,0,257091700,Moored,-128.0,...,360.0,511,NotAvailable,True,2256,0,BATSERVICE V,,60,2024-04-14T14:03:12.273384
441644,2024,4,15,2024-04-14 14:03:14.309100,53619976,1,0,258593000,UnderWayUsingEngine,0.0,...,40.2,54,NotAvailable,False,23564,9383388,FJORDDROTT,,40,2024-04-14T14:03:14.309100


## Convert to GeoDataFrame

In [5]:
%%time
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.long, df.lat), crs="EPSG:4326")

CPU times: user 71.4 ms, sys: 16.3 ms, total: 87.7 ms
Wall time: 87.1 ms


## Visualize as points

In [6]:
%%time
viz(gdf[["geometry", "name", "mmsi", "status", "course", "speed", "timestamp_str"]])

CPU times: user 440 ms, sys: 79.2 ms, total: 519 ms
Wall time: 228 ms


## Visualize as tracks

In [7]:
def to_linestring(group):
    coords = list(group.geometry)
    # Need at least 2 points for a line
    if len(coords) >= 2:
        return LineString(coords)
    else:
        return None


gdf_linestring = (gdf
 .sort_values("timestamp_str")
 .groupby(["mmsi", "name"])
 .apply(lambda g: pd.Series(
     {"geometry": to_linestring(g)})

 )
)

/tmp/ipykernel_16026/3178371509.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series(


In [8]:
viz(gdf_linestring)

/home/knuthp/src/spartid-ais/.venv/lib/python3.13/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


In [9]:
layer = PathLayer.from_geopandas(
    gdf=gdf_linestring,
    pickable=True,
    auto_highlight=True,
    get_color=[255, 150, 150],
    highlight_color=[255, 0, 0, 200],
    width_min_pixels=2
)
map = Map(layers=[layer])
map

/home/knuthp/src/spartid-ais/.venv/lib/python3.13/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


In [10]:
map.to_html("pathlayer.html")

## Visualize movement - Tripslayer

In [11]:
gdf.dtypes

year                     int64[pyarrow]
month                    int64[pyarrow]
week                     int64[pyarrow]
timestamp        timestamp[us][pyarrow]
id                       int32[pyarrow]
msg_type                 int16[pyarrow]
repeat                   int16[pyarrow]
mmsi                     int32[pyarrow]
status                  string[pyarrow]
turn                    double[pyarrow]
speed                   double[pyarrow]
accuracy                  bool[pyarrow]
lat                     double[pyarrow]
long                    double[pyarrow]
course                  double[pyarrow]
heading                  int32[pyarrow]
maneuver                string[pyarrow]
raim                      bool[pyarrow]
radio                    int32[pyarrow]
imo                              object
name                             object
flag                             object
type                             object
timestamp_str                    object
geometry                       geometry


In [12]:
gdf_live = (
    gdf[["geometry", "mmsi", "timestamp_str", "timestamp"]]
#    .query("mmsi in ['257514900']")
)
traj_collection = mpd.TrajectoryCollection(gdf_live, "mmsi", t="timestamp")
print(len(traj_collection))

193


In [13]:
gdf_live

# traj_collection = mpd.MinTimeDeltaGeneralizer(traj_collection).generalize(
#     tolerance=timedelta(minutes=1),
# )
# print(len(traj_collection))

,geometry,mmsi,timestamp_str,timestamp
0,POINT (10.64286 59.22359),257935000,2024-04-08T00:00:00.664767,2024-04-08 00:00:00.664767
1,POINT (10.47066 59.24466),257556800,2024-04-08T00:00:00.677092,2024-04-08 00:00:00.677092
2,POINT (10.42318 59.25984),257058500,2024-04-08T00:00:03.832583,2024-04-08 00:00:03.832583
3,POINT (10.4577 59.23665),257319700,2024-04-08T00:00:03.843323,2024-04-08 00:00:03.843323
4,POINT (10.47324 59.24826),257173700,2024-04-08T00:00:04.801238,2024-04-08 00:00:04.801238
...,...,...,...,...
441641,POINT (10.65367 59.42966),259402000,2024-04-14T14:03:11.250461,2024-04-14 14:03:11.250461
441642,POINT (10.41254 59.61233),258010350,2024-04-14T14:03:11.257431,2024-04-14 14:03:11.257431
441643,POINT (10.73136 59.91068),257091700,2024-04-14T14:03:12.273384,2024-04-14 14:03:12.273384
441644,POINT (10.58634 59.73383),258593000,2024-04-14T14:03:14.309100,2024-04-14 14:03:14.309100


In [14]:
# traj_collection.add_speed(overwrite=True).add_distance(overwrite=True)
#traj_collection.trajectories[100].df["mmsi"]
traj_collection.add_timedelta(overwrite=True)
traj_collection.add_distance(overwrite=True)
traj_collection.trajectories[0].df["distance"].sort_values()

timestamp
2024-04-08 00:01:48.304604    0.000000
2024-04-11 22:37:29.491290    0.000000
2024-04-12 04:18:41.985984    0.000000
2024-04-08 13:40:39.316623    0.000000
2024-04-09 20:49:48.139299    0.000000
                                ...   
2024-04-13 08:32:42.820181    6.158494
2024-04-10 06:19:26.809376    6.366167
2024-04-10 02:13:11.237588    6.782629
2024-04-12 08:45:41.667479    7.157456
2024-04-08 19:09:58.003071    7.247041
Name: distance, Length: 3314, dtype: float64

In [15]:
trips_layer = TripsLayer.from_movingpandas(
    traj_collection,
    # get_color=get_color,
    width_min_pixels=5,
    trail_length=500,
)
linestring_layer = PathLayer(
    table=trips_layer.table,
    # get_color=get_color,
    width_min_pixels=1,
    opacity=0.005,
)

m_trips = Map(trips_layer, height=600)
# m_trips.add_layer(linestring_layer)
m_trips

/home/knuthp/src/spartid-ais/.venv/lib/python3.13/site-packages/lonboard/traits/_timestamp.py:152: UserWarning: Reducing precision of input timestamp data to 's' to fit into available GPU precision.
  warnings.warn(


In [16]:
trips_layer.animate(step=timedelta(seconds=160), fps=10)

In [17]:
m_trips.to_html("triplayer.html")